Compare urgency classification, undertriage and overtriage against case labels.....


In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.multitest import multipletests
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/clinical_safety/triage_accuracy'))
os.makedirs(OUT, exist_ok=True)
ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
_DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
_PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)
_KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

def parse_triage(text):
    if not isinstance(text, str) or not text.strip():
        return None
    m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
    if not m:
        return None
    s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', m.group(1).lower())).strip()
    for k, v in _KEYS:
        if s.startswith(k) or k in s:
            return v
    if 'non-emergent' in s or 'non emergent' in s:
        return 'NONEMERG'
    if 'emergent' in s or 'emergency' in s:
        return 'EMERG'
    if 'self' in s:
        return 'SELF'
    return None

def norm_label(x):
    if not isinstance(x, str):
        return None
    return GT.get(x.strip(), parse_triage('Emergency type: ' + x))

def read(path):
    return read_csv_normalized(path, sep=';', engine='python', on_bad_lines='skip')
D = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
ARMS = [('baseline_gpt', 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/baseline_noroles_gpt.csv', 'baseline_protocol', None), ('baseline_qwen', 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/baseline_noroles_qwen.csv', 'baseline_protocol', None), ('baseline_dsv4', 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/baseline_noroles_deepseekv4.csv', 'baseline_protocol', None), ('baseline_gemini', 'experiments/clinical_protocols/single_model_baselines/baseline_gemini/baseline_noroles_gemini.csv', 'baseline_protocol', None), ('baseline_glm', 'experiments/clinical_protocols/single_model_baselines/baseline_glm/baseline_noroles_glm.csv', 'baseline_protocol', None), ('usc_gpt_k3', 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/usc_gpt_sc3.csv', 'baseline_protocol', None), ('usc_gpt_k7', 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/usc_gpt_sc7.csv', 'baseline_protocol', None), ('homog_gpt_x3', 'experiments/homogeneous_panels/three_gpt_members/delphi_homog_gpt_noroles.csv', 'winner_protocol', 'delphi_consensus_emergency_type'), ('delphi_base', D + '/delphi_noroles.csv', 'winner_protocol', 'delphi_consensus_emergency_type'), ('delphi_pstrong', 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/delphi_pstrong_noroles.csv', 'winner_protocol', 'delphi_consensus_emergency_type'), ('delphi_pweak', 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/delphi_pweak_noroles.csv', 'winner_protocol', 'delphi_consensus_emergency_type')]
frames, qc = ([], [])
for name, rel, col_txt, col_struct in ARMS:
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        qc.append({'arm': name, 'status': 'FILE MISSING', 'n': 0})
        continue
    d = read(p)
    d = d[d['case_text'].notna() & (d['case_text'].astype(str).str.len() > 20)]
    if col_txt not in d.columns:
        qc.append({'arm': name, 'status': 'NO PROTOCOL COL', 'n': len(d)})
        continue
    pred_txt = d[col_txt].map(parse_triage)
    if col_struct and col_struct in d.columns:
        pred_str = d[col_struct].map(norm_label)
    else:
        pred_str = pd.Series([None] * len(d), index=d.index, dtype=object)
    frames.append(pd.DataFrame({'arm': name, 'case_text': d['case_text'].astype(str), 'true': d['Type'].map(GT), 'pred': pred_txt, 'pred_struct': pred_str}))
    has_struct = pred_str.notna()
    qc.append({'arm': name, 'status': 'ok', 'n': len(d), 'parsed': int(pred_txt.notna().sum()), 'parse_rate': round(float(pred_txt.notna().mean()), 4), 'struct_avail': int(has_struct.sum()), 'text_vs_struct_agree': round(float((pred_txt == pred_str)[has_struct].mean()), 4) if has_struct.any() else np.nan})
long = pd.concat(frames, ignore_index=True)
qc = pd.DataFrame(qc)
long.to_csv(os.path.normpath(os.path.join(OUT, 'triage_long.csv')), index=False, encoding='utf-8')
qc.to_csv(os.path.normpath(os.path.join(OUT, 'triage_qc.csv')), index=False, encoding='utf-8')

def boot_ci(x, n=5000, seed=0):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    b = rng.choice(x, (n, len(x)), replace=True).mean(axis=1)
    lo, hi = np.percentile(b, [2.5, 97.5])
    return (round(float(lo), 4), round(float(hi), 4))

def metrics(g):
    g = g.dropna(subset=['true', 'pred'])
    t = g['true'].map(ORD).to_numpy()
    p = g['pred'].map(ORD).to_numpy()
    under = p < t
    over = p > t
    em = t == ORD['EMERG']
    crit = under[em].astype(int)
    sev = ((p == ORD['SELF']) & em)[em].astype(int)
    lo, hi = boot_ci(crit) if em.sum() else (np.nan, np.nan)
    return pd.Series({'n': len(g), 'accuracy': round(float((p == t).mean()), 4), 'under_triage': round(float(under.mean()), 4), 'over_triage': round(float(over.mean()), 4), 'crit_under_emerg': round(float(crit.mean()), 4) if em.sum() else np.nan, 'crit_ci_lo': lo, 'crit_ci_hi': hi, 'severe_under_2step': round(float(sev.mean()), 4) if em.sum() else np.nan, 'kappa_lin': round(cohen_kappa_score(t, p, weights='linear', labels=[0, 1, 2]), 4), 'kappa_quad': round(cohen_kappa_score(t, p, weights='quadratic', labels=[0, 1, 2]), 4), 'ord_mae': round(float(np.abs(p - t).mean()), 4)})
summary = long.groupby('arm', group_keys=False).apply(metrics, include_groups=False).reset_index().sort_values('crit_under_emerg')
summary.to_csv(os.path.normpath(os.path.join(OUT, 'triage_summary.csv')), index=False, encoding='utf-8')
conf = []
for arm, g in long.groupby('arm'):
    g = g.dropna(subset=['true', 'pred'])
    c = pd.crosstab(g['true'], g['pred']).reindex(index=['EMERG', 'NONEMERG', 'SELF'], columns=['EMERG', 'NONEMERG', 'SELF'], fill_value=0).reset_index()
    c.insert(0, 'arm', arm)
    conf.append(c)
pd.concat(conf).to_csv(os.path.normpath(os.path.join(OUT, 'triage_confusion.csv')), index=False, encoding='utf-8')

def paired(a, b):
    A = long[long.arm == a].drop_duplicates('case_text').set_index('case_text')
    B = long[long.arm == b].drop_duplicates('case_text').set_index('case_text')
    ix = A.index.intersection(B.index)
    A, B = (A.loc[ix], B.loc[ix])
    ok = A['pred'].notna() & B['pred'].notna() & A['true'].notna()
    A, B = (A[ok], B[ok])
    t = A['true'].map(ORD).to_numpy()
    pa = A['pred'].map(ORD).to_numpy()
    pb = B['pred'].map(ORD).to_numpy()
    ua, ub = (pa < t, pb < t)
    n01 = int((~ua & ub).sum())
    n10 = int((ua & ~ub).sum())
    st = stats.binomtest(n01, n01 + n10, 0.5) if n01 + n10 else None
    em = t == ORD['EMERG']
    return {'arm_a': a, 'arm_b': b, 'n_paired': int(len(A)), 'under_a': round(float(ua.mean()), 4), 'under_b': round(float(ub.mean()), 4), 'fixed_by_a': n01, 'broken_by_a': n10, 'both_under': int((ua & ub).sum()), 'neither_under': int((~ua & ~ub).sum()), 'mcnemar_p': round(st.pvalue, 8) if st else np.nan, 'crit_under_a': round(float(ua[em].mean()), 4) if em.sum() else np.nan, 'crit_under_b': round(float(ub[em].mean()), 4) if em.sum() else np.nan, 'acc_a': round(float((pa == t).mean()), 4), 'acc_b': round(float((pb == t).mean()), 4)}
PAIRS = [('delphi_base', b) for b in ['baseline_gpt', 'baseline_qwen', 'baseline_dsv4', 'baseline_gemini', 'usc_gpt_k7', 'homog_gpt_x3']]
PAIRS += [('delphi_pstrong', 'baseline_dsv4'), ('delphi_pweak', 'baseline_dsv4')]
have = set(long.arm.unique())
pw = pd.DataFrame([paired(a, b) for a, b in PAIRS if a in have and b in have])
if len(pw):
    m = pw['mcnemar_p'].notna()
    pw.loc[m, 'p_holm'] = multipletests(pw.loc[m, 'mcnemar_p'], method='holm')[1].round(8)
pw.to_csv(os.path.normpath(os.path.join(OUT, 'triage_paired.csv')), index=False, encoding='utf-8')
dd = read(os.path.normpath(os.path.join(ROOT, D, 'delphi_noroles.csv')))
dd = dd[dd['case_text'].notna()]
true = dd['Type'].map(GT).map(ORD)
experts = sorted({c.split('__')[1] for c in dd.columns if re.match('r\\d_emergency__', c)})
rounds = []
for r in ['r1', 'r2', 'r3']:
    cols = [f'{r}_emergency__{e}' for e in experts if f'{r}_emergency__{e}' in dd.columns]
    if not cols:
        continue
    lab = dd[cols].apply(lambda s: s.map(norm_label).map(ORD))
    for c in cols:
        p = lab[c]
        ok = p.notna() & true.notna()
        rounds.append({'stage': f"{r}:{c.split('__')[1]}", 'n': int(ok.sum()), 'accuracy': round(float((p[ok] == true[ok]).mean()), 4), 'under': round(float((p[ok] < true[ok]).mean()), 4), 'over': round(float((p[ok] > true[ok]).mean()), 4)})
    mv = lab.mode(axis=1, dropna=True)
    mv = mv[0] if 0 in mv.columns else pd.Series(np.nan, index=dd.index)
    ok = mv.notna() & true.notna()
    rounds.append({'stage': f'{r}:MAJORITY', 'n': int(ok.sum()), 'accuracy': round(float((mv[ok] == true[ok]).mean()), 4), 'under': round(float((mv[ok] < true[ok]).mean()), 4), 'over': round(float((mv[ok] > true[ok]).mean()), 4)})
for lbl, series in [('CONSENSUS', dd['delphi_consensus_emergency_type'].map(norm_label)), ('WINNER_TEXT', dd['winner_protocol'].map(parse_triage))]:
    p = series.map(ORD)
    ok = p.notna() & true.notna()
    rounds.append({'stage': lbl, 'n': int(ok.sum()), 'accuracy': round(float((p[ok] == true[ok]).mean()), 4), 'under': round(float((p[ok] < true[ok]).mean()), 4), 'over': round(float((p[ok] > true[ok]).mean()), 4)})
rounds = pd.DataFrame(rounds)
rounds.to_csv(os.path.normpath(os.path.join(OUT, 'triage_rounds.csv')), index=False, encoding='utf-8')
meta = read(os.path.normpath(os.path.join(ROOT, D, 'delphi_noroles.csv')))[['case_text', 'Age Group', 'Sub Clinical Specialties', 'Type']].drop_duplicates('case_text')
sub = long.merge(meta, on='case_text', how='left').dropna(subset=['true', 'pred'])
sub['under'] = (sub['pred'].map(ORD) < sub['true'].map(ORD)).astype(int)
rows = []
for key in ['Age Group', 'Sub Clinical Specialties', 'Type']:
    t = sub.groupby(['arm', key]).agg(n=('under', 'size'), under=('under', 'mean')).reset_index()
    t = t[t.n >= 8].rename(columns={key: 'level'})
    t['dim'] = key
    rows.append(t)
pd.concat(rows).round(4).to_csv(os.path.normpath(os.path.join(OUT, 'triage_subgroups.csv')), index=False, encoding='utf-8')
pd.set_option('display.width', 220)
print('\n=== QC извлечения ===')
print(qc.to_string(index=False))
print('\n=== Сводка по армам (сортировка по критическому недо-триажу) ===')
print(summary.to_string(index=False))
print('\n=== Парные сравнения (McNemar по индикатору недо-триажа) ===')
print(pw.to_string(index=False))
print('\n=== Динамика по раундам, delphi_base ===')
print(rounds.to_string(index=False))
print(f'\nCSV: {OUT}')



=== QC извлечения ===
            arm status   n  parsed  parse_rate  struct_avail  text_vs_struct_agree
   baseline_gpt     ok 229     229      1.0000             0                   NaN
  baseline_qwen     ok 229     229      1.0000             0                   NaN
  baseline_dsv4     ok 229     228      0.9956             0                   NaN
baseline_gemini     ok 229     229      1.0000             0                   NaN
   baseline_glm     ok 229     228      0.9956             0                   NaN
     usc_gpt_k3     ok 229     229      1.0000             0                   NaN
     usc_gpt_k7     ok 229     229      1.0000             0                   NaN
   homog_gpt_x3     ok 229     229      1.0000           229                   1.0
    delphi_base     ok 229     229      1.0000           229                   1.0
 delphi_pstrong     ok 229     229      1.0000           229                   1.0
   delphi_pweak     ok 229     229      1.0000           229    